In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l1, l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')

In [2]:
pwd

'C:\\Users\\admin\\OneDrive\\Documents\\deep learing'

In [3]:
# Load dataset
df = pd.read_csv("Boston-house-price-data.csv")

In [4]:
# Split
X = df.drop("MEDV", axis=1)
y = df["MEDV"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [10]:
!pip install optuna

  Using cached optuna-4.9.0-py3-none-any.whl.metadata (15 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
  Using cached mako-1.3.12-py3-none-any.whl.metadata (2.9 kB)
Using cached optuna-4.9.0-py3-none-any.whl (425 kB)
Using cached alembic-1.18.4-py3-none-any.whl (263 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 799.2 kB/s eta 0:00:03
   -------------- ------------------------- 0.8/2.1 MB 819.2 kB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 1.0 MB/s eta 0:00:02
   ------------------------ --------------- 1.3/2.1 MB 1.0 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 M

In [13]:

import optuna

In [16]:
import numpy as np

print("Unique labels:", np.unique(y_train))
print("Number of classes:", len(np.unique(y_train)))

Unique labels: [ 5.   5.6  6.3  7.2  7.4  7.5  8.1  8.3  8.4  8.5  8.7  8.8  9.5 10.2
 10.4 10.5 10.9 11.  11.3 11.5 11.7 11.8 11.9 12.  12.1 12.3 12.5 12.6
 12.7 12.8 13.  13.1 13.2 13.3 13.4 13.5 13.6 13.8 13.9 14.1 14.3 14.4
 14.5 14.6 14.8 14.9 15.  15.2 15.3 15.4 15.6 15.7 16.  16.1 16.2 16.3
 16.5 16.6 16.7 16.8 17.  17.1 17.3 17.4 17.5 17.6 17.7 17.8 18.  18.1
 18.2 18.3 18.4 18.5 18.6 18.7 18.8 18.9 19.  19.1 19.2 19.3 19.4 19.5
 19.6 19.7 19.8 19.9 20.  20.1 20.2 20.3 20.4 20.5 20.6 20.7 20.8 21.
 21.1 21.2 21.4 21.5 21.6 21.7 21.8 21.9 22.  22.1 22.2 22.3 22.4 22.5
 22.6 22.7 22.8 22.9 23.  23.1 23.2 23.3 23.4 23.5 23.6 23.7 23.8 23.9
 24.  24.1 24.3 24.4 24.5 24.6 24.7 24.8 25.  25.1 25.3 26.2 26.4 26.6
 26.7 27.  27.1 27.5 27.9 28.  28.1 28.4 28.6 28.7 29.  29.1 29.4 29.6
 29.8 29.9 30.1 30.3 30.5 30.7 31.  31.1 31.2 31.5 31.6 31.7 32.  32.2
 32.5 32.7 32.9 33.  33.1 33.2 33.3 33.4 33.8 34.6 34.9 35.1 35.2 36.
 36.1 36.2 36.4 36.5 37.  37.2 37.3 37.6 37.9 38.7 39.8 41.3 41.

In [17]:
import optuna
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam

def objective(trial):

    model = Sequential()

    # Input Layer
    model.add(Input(shape=(X_train.shape[1],)))

    # Hidden Layers
    n_layers = trial.suggest_int('n_layers', 1, 3)

    for i in range(n_layers):

        units = trial.suggest_int(
            f'units_{i}',
            16,
            128
        )

        model.add(
            Dense(
                units,
                activation='relu'
            )
        )

        dropout_rate = trial.suggest_float(
            f'dropout_{i}',
            0.2,
            0.5
        )

        model.add(
            Dropout(dropout_rate)
        )

    # Regression Output Layer
    model.add(Dense(1))

    # Learning Rate
    lr = trial.suggest_float(
        'lr',
        1e-4,
        1e-2,
        log=True
    )

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='mse',
        metrics=['mae']
    )

    history = model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        epochs=50,
        batch_size=trial.suggest_categorical(
            'batch_size',
            [16, 32, 64]
        ),
        verbose=0
    )

    # Minimize validation loss
    return min(history.history['val_loss'])

In [18]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

[I 2026-06-25 09:28:19,906] A new study created in memory with name: no-name-2b46a4bc-dd61-4b17-bcfc-d15a44556ff5
[I 2026-06-25 09:28:37,898] Trial 0 finished with value: 31.136842727661133 and parameters: {'n_layers': 1, 'units_0': 26, 'dropout_0': 0.4519895126874002, 'lr': 0.0021742306272150074, 'batch_size': 16}. Best is trial 0 with value: 31.136842727661133.
[I 2026-06-25 09:28:55,044] Trial 1 finished with value: 218.04600524902344 and parameters: {'n_layers': 2, 'units_0': 24, 'dropout_0': 0.2542774545461642, 'units_1': 69, 'dropout_1': 0.4615658301477495, 'lr': 0.0001525293615033252, 'batch_size': 32}. Best is trial 0 with value: 31.136842727661133.
[I 2026-06-25 09:29:12,871] Trial 2 finished with value: 19.302658081054688 and parameters: {'n_layers': 2, 'units_0': 58, 'dropout_0': 0.28419813884187267, 'units_1': 58, 'dropout_1': 0.2164511503802476, 'lr': 0.0004360582719239922, 'batch_size': 16}. Best is trial 2 with value: 19.302658081054688.
[I 2026-06-25 09:29:30,052] Trial

## Best Hyperparameters Found

In [21]:
print('Best MAE:', study.best_value)
print('Best Parameters:', study.best_params)

Best MAE: 9.319525718688965
Best Parameters: {'n_layers': 3, 'units_0': 109, 'dropout_0': 0.31995207426004096, 'units_1': 107, 'dropout_1': 0.49713919031303744, 'units_2': 63, 'dropout_2': 0.20135267460492112, 'lr': 0.005809321563987487, 'batch_size': 16}


## Train Final ANN using Best Parameters

In [22]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam

best_params = study.best_params

model = Sequential()

# Input layer
model.add(Input(shape=(X_train.shape[1],)))

# Hidden layers
for i in range(best_params['n_layers']):
    model.add(Dense(best_params[f'units_{i}'],activation='relu'))

    model.add( Dropout( best_params[f'dropout_{i}']))

# Regression output layer
model.add(Dense(1))
# Compile
model.compile(
    optimizer=Adam(
        learning_rate=best_params['lr']
    ),
    loss='mse',
    metrics=['mae']
)

# Train
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=best_params['batch_size'],
    validation_split=0.2,
    verbose=1
)

Epoch 1/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - loss: 226.9840 - mae: 11.5952 - val_loss: 80.6772 - val_mae: 7.3700
Epoch 2/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 68.2435 - mae: 6.0349 - val_loss: 36.0769 - val_mae: 4.6596
Epoch 3/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 53.9150 - mae: 5.4507 - val_loss: 22.3762 - val_mae: 3.4700
Epoch 4/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 47.3757 - mae: 5.2197 - val_loss: 22.6802 - val_mae: 3.2517
Epoch 5/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 43.4475 - mae: 4.9895 - val_loss: 19.5245 - val_mae: 3.3448
Epoch 6/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 49.6514 - mae: 5.0943 - val_loss: 30.5604 - val_mae: 3.8972
Epoch 7/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 46.4522 - mae: 5.1202 - val_loss: 18.2412 - val_mae: 3.0846
Epoch 8/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 44.3134 - mae: 5.0960 - val_loss: 20.7820 - val_mae: 3.1215
Epoch 9/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/

In [23]:
loss, mae = model.evaluate(X_test, y_test)

print("Test MSE:", loss)
print("Test MAE:", mae)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 12.5619 - mae: 2.3799
Test MSE: 12.56188678741455
Test MAE: 2.379920244216919


In [24]:
from sklearn.metrics import r2_score

# Predictions
y_pred = model.predict(X_test, verbose=0)

# Calculate R² Score
r2 = r2_score(y_test, y_pred)

print("R² Score:", r2)

R² Score: 0.8287026000011823


In [25]:

y_train_pred = model.predict(X_train, verbose=0)
y_test_pred = model.predict(X_test, verbose=0)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Train R² Score: {train_r2:.4f}")
print(f"Test  R² Score: {test_r2:.4f}")

Train R² Score: 0.8970
Test  R² Score: 0.8287
